# Intermediate 03 lab — From document pixels to structured evidence

This lab builds a transparent document-intelligence pipeline for mixed enterprise invoices, inspection reports, and maintenance forms. The central invariant is:

> A transformation may add a representation, but it must not sever lineage to the original document.

The default path uses deterministic synthetic pages and explicitly labeled local proxies. It is not an OCR, layout-model, or document-foundation-model benchmark.

![A document becomes structured evidence while provenance survives every transformation.](assets/document-intelligence-pipeline.svg)


## 1. Scenario, source contract, and safety boundary

- **Template A / construction:** implement and debug primitives.
- **Template B / development:** select rules and review policy.
- **Template C / held-out test:** reporting only; never tune a threshold, parser, field rule, or template decision.
- Documents are split before pages, OCR records, table cells, or perturbations are derived.
- Generated documents and artifacts stay under `.artifacts/`, which is gitignored.
- Outputs are advisory. `authorization = "none"`; no financial or maintenance action is permitted.
- Document text is untrusted data, including strings that resemble instructions.


In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont

SEED = 20260907
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 90)

PAGE_WIDTH, PAGE_HEIGHT = 800, 1000
LOCAL_OCR_LABEL = "local_ocr_proxy"
LOCAL_LAYOUT_LABEL = "local_layout_proxy"
FIELD_STATES = {"verified", "uncertain", "missing", "conflicting", "unsupported"}
SOURCE_CONTRACT = {
    "Template A": "construction",
    "Template B": "development",
    "Template C": "test_reporting_only",
}
DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this notebook runtime only."

print({
    "seed": SEED,
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": Image.__version__,
    "matplotlib": matplotlib.__version__,
    "source_contract": SOURCE_CONTRACT,
})


## 2. Page and geometry contracts

A box must name its format, unit, origin, page dimensions, and rendering DPI. PDF points and image pixels are different frames; a visually aligned overlay is not evidence that the numerical transform is correct.


In [ ]:
@dataclass(frozen=True)
class PageContract:
    document_id: str
    page: int
    width: int
    height: int
    dpi: int
    origin: str = "top_left"
    unit: str = "pixel"


@dataclass(frozen=True)
class RegionRecord:
    region_id: str
    page: int
    kind: str
    box: tuple[float, float, float, float]
    reading_order: int
    column: int = 0
    text: str = ""


def validate_xyxy(box, width: float, height: float) -> bool:
    x1, y1, x2, y2 = map(float, box)
    return 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height


def pixel_to_normalized_box(box, page_width: int, page_height: int):
    if not validate_xyxy(box, page_width, page_height):
        raise ValueError("pixel xyxy box is outside the page contract")
    x1, y1, x2, y2 = map(float, box)
    return (x1 / page_width, y1 / page_height, x2 / page_width, y2 / page_height)


def normalized_to_pixel_box(box, page_width: int, page_height: int):
    x1, y1, x2, y2 = map(float, box)
    if not (0 <= x1 < x2 <= 1 and 0 <= y1 < y2 <= 1):
        raise ValueError("normalized xyxy box must be inside [0, 1]")
    return (x1 * page_width, y1 * page_height, x2 * page_width, y2 * page_height)


def pdf_points_to_pixels(box, dpi: int, page_height_points: float | None = None, origin="top_left"):
    scale = dpi / 72.0
    x1, y1, x2, y2 = map(float, box)
    if origin == "top_left":
        return tuple(value * scale for value in (x1, y1, x2, y2))
    if origin == "bottom_left" and page_height_points is not None:
        return (x1 * scale, (page_height_points - y2) * scale,
                x2 * scale, (page_height_points - y1) * scale)
    raise ValueError("bottom-left conversion requires page_height_points")


def route_document_input(kind: str, embedded_text_chars: int = 0, image_region_fraction: float = 0.0) -> dict:
    if kind == "digital_pdf" and embedded_text_chars > 0 and image_region_fraction < 0.2:
        return {"route": "embedded_text_plus_render", "ocr": "only_flagged_regions"}
    if kind == "scanned_pdf" or kind == "raster_image":
        return {"route": "render_then_ocr_layout", "ocr": "required"}
    if kind == "hybrid_pdf" or (embedded_text_chars > 0 and image_region_fraction >= 0.2):
        return {"route": "reconcile_text_and_image_regions", "ocr": "region_specific"}
    return {"route": "quarantine_or_review", "ocr": "not_started"}


example_box = (120, 82, 234, 106)
normalized = pixel_to_normalized_box(example_box, 1200, 1600)
round_trip = normalized_to_pixel_box(normalized, 1200, 1600)
assert np.allclose(example_box, round_trip)
assert np.allclose(pdf_points_to_pixels((72, 72, 144, 144), 144), (144, 144, 288, 288))
assert np.allclose(pdf_points_to_pixels((72, 72, 144, 144), 144, 792, "bottom_left"), (144, 1296, 288, 1440))
assert route_document_input("digital_pdf", embedded_text_chars=1200)["ocr"] == "only_flagged_regions"
assert route_document_input("scanned_pdf")["ocr"] == "required"
{"normalized": normalized, "round_trip": round_trip, "routing_examples": [
    route_document_input("digital_pdf", embedded_text_chars=1200), route_document_input("scanned_pdf"), route_document_input("hybrid_pdf", 800, .4)]}


## 3. Rendering DPI experiment

Higher DPI makes a small glyph occupy more pixels, but page pixels grow quadratically with linear resolution. The legibility signal below is a declared geometric proxy—not measured OCR accuracy.


In [ ]:
def dpi_cost(width_in=8.5, height_in=11.0, font_points=7.0):
    rows = []
    for dpi in (72, 150, 300):
        width, height = round(width_in * dpi), round(height_in * dpi)
        glyph_height = font_points * dpi / 72
        rows.append({
            "dpi": dpi,
            "page_megapixels": width * height / 1e6,
            "approx_rgb_mib": width * height * 3 / (1024 ** 2),
            "seven_point_glyph_px": glyph_height,
            "small_text_proxy_legible": glyph_height >= 12,
        })
    return pd.DataFrame(rows)


dpi_results = dpi_cost()
ax = dpi_results.plot(x="dpi", y=["page_megapixels", "seven_point_glyph_px"], marker="o", secondary_y="seven_point_glyph_px", figsize=(8, 4))
ax.set_title("Rendering cost and small-glyph sampling")
ax.set_ylabel("page megapixels")
plt.tight_layout()
dpi_results


## 4. Generate a mixed, source-isolated corpus

Each document has two pages, typed regions, word boxes, fields, a span-aware table, checkboxes, a figure/caption pair, and a continued table. The renderer creates pixels from the same structured source, allowing exact failure injection without pretending the proxy inferred ground truth.


In [ ]:
TEMPLATE_STYLE = {
    "Template A": {"accent": (37, 91, 153), "paper": (250, 251, 253), "key_x": 65, "value_x": 275, "columns": 1},
    "Template B": {"accent": (24, 125, 118), "paper": (249, 248, 242), "key_x": 70, "value_x": 300, "columns": 2},
    "Template C": {"accent": (126, 78, 145), "paper": (244, 246, 241), "key_x": 420, "value_x": 110, "columns": 2},
}
DOC_FIELDS = {
    "invoice": [("invoice_number", "Invoice Number"), ("purchase_order", "Purchase Order Number"), ("date", "Invoice Date"), ("supplier", "Supplier")],
    "inspection_report": [("report_id", "Report ID"), ("date", "Inspection Date"), ("site", "Site"), ("score", "Quality Score")],
    "maintenance_form": [("work_order", "Work Order"), ("date", "Service Date"), ("asset", "Asset ID"), ("status", "Status")],
}


def canonical_hash(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, separators=(",", ":")).encode()).hexdigest()


def field_values(doc_type: str, serial: int):
    if doc_type == "invoice":
        return {"invoice_number": f"INV-{1040 + serial}", "purchase_order": f"PO-{870 + serial}", "date": "09/07/26", "supplier": "Northstar Components", "invoice_total": "$1,240.00"}
    if doc_type == "inspection_report":
        return {"report_id": f"RPT-{210 + serial}", "date": "2026-09-07", "site": "Plant 4", "score": "97.5%"}
    return {"work_order": f"WO-{330 + serial}", "date": "Sept. 7, 2026", "asset": "PUMP-17", "status": "Requires review"}


def add_text_token(tokens, page, text, box, role, field_name=None, font_size=12, column=0):
    token_id = f"p{page}-w{len(tokens) + 1:03d}"
    tokens.append({"word_id": token_id, "page": page, "text": text, "box": tuple(box), "role": role,
                   "field_name": field_name, "font_size": font_size, "column": column})
    return token_id


def make_table(page: int, table_id: str, start_row: int, continued: bool):
    xs = [70, 360, 555, 730]
    y0, row_h = 500 if page == 1 else 150, 42
    cells = []
    if not continued:
        cells.append({"cell_id": f"{table_id}-quarter", "page": page, "row_start": 0, "row_end": 0,
                      "col_start": 0, "col_end": 2, "role": "super_header", "text": "Q1 service ledger",
                      "box": (xs[0], y0, xs[-1], y0 + row_h)})
        header_row = 1
        header_offset = 1
    else:
        header_row = start_row
        header_offset = 0
    for col, text in enumerate(("Item", "Units", "Amount")):
        cells.append({"cell_id": f"{table_id}-r{header_row}-c{col}", "page": page, "row_start": header_row,
                      "row_end": header_row, "col_start": col, "col_end": col, "role": "column_header",
                      "text": text, "box": (xs[col], y0 + header_offset * row_h, xs[col + 1], y0 + (header_offset + 1) * row_h)})
    base = header_row + 1
    values = [("Seal kit", "2", "$440.00"), ("Inspection", "4", "$800.00")] if page == 1 else [("Adjustment", "1", "$0.00"), ("Total", "", "$1,240.00")]
    for r_off, row in enumerate(values):
        for col, text in enumerate(row):
            row_i = base + r_off
            top = y0 + (header_offset + 1 + r_off) * row_h
            cells.append({"cell_id": f"{table_id}-r{row_i}-c{col}", "page": page, "row_start": row_i,
                          "row_end": row_i, "col_start": col, "col_end": col, "role": "body",
                          "text": text, "box": (xs[col], top, xs[col + 1], top + row_h)})
    return {"table_id": table_id, "page": page, "continued": continued, "columns": 3, "cells": cells,
            "box": (xs[0], y0, xs[-1], max(cell["box"][3] for cell in cells))}


def make_document(template: str, doc_type: str, serial: int):
    style = TEMPLATE_STYLE[template]
    doc_id = f"{template.lower().replace(' ', '-')}-{doc_type}-{serial:02d}"
    values = field_values(doc_type, serial)
    pages = []
    all_regions = []
    all_tokens = []
    all_tables = []
    for page_number in (1, 2):
        tokens, regions = [], []
        if page_number == 1:
            title = doc_type.replace("_", " ").title()
            add_text_token(tokens, 1, title, (55, 55, 390, 90), "title", font_size=24)
            regions.append(RegionRecord(f"{doc_id}:p1:title", 1, "title", (45, 40, 750, 105), 0, text=title))
            for i, (field_name, label) in enumerate(DOC_FIELDS[doc_type]):
                row = i % 2 if style["columns"] == 2 else i
                col = i // 2 if style["columns"] == 2 else 0
                y = 145 + row * 72
                if template == "Template C":
                    kx, vx = style["key_x"] + col * 30, style["value_x"] + col * 275
                else:
                    kx, vx = style["key_x"] + col * 385, style["value_x"] + col * 385
                add_text_token(tokens, 1, label, (kx, y, min(kx + 190, 770), y + 24), "key", field_name, 11, col)
                add_text_token(tokens, 1, values[field_name], (vx, y, min(vx + 190, 775), y + 24), "value", field_name, 11, col)
                regions.append(RegionRecord(f"{doc_id}:p1:field:{field_name}", 1, "form_field",
                                            (min(kx, vx) - 8, y - 8, min(max(kx + 190, vx + 190), 780), y + 32), 1 + i, col, label))
            paragraphs = [("Scope and observations", 340, 0), ("Corrective action and notes", 400, 1 if style["columns"] == 2 else 0)]
            for j, (text, y, col) in enumerate(paragraphs):
                x = 55 + col * 385
                add_text_token(tokens, 1, text, (x, y, min(x + 320, 760), y + 26), "paragraph", font_size=10, column=col)
                regions.append(RegionRecord(f"{doc_id}:p1:paragraph:{j}", 1, "paragraph", (x - 5, y - 8, min(x + 330, 770), y + 42), 10 + j, col, text))
            table = make_table(1, f"{doc_id}:ledger", 0, False)
            all_tables.append(table)
            regions.append(RegionRecord(f"{doc_id}:p1:table", 1, "table", table["box"], 20, text="service ledger"))
            for cell in table["cells"]:
                if cell["text"]:
                    add_text_token(tokens, 1, cell["text"], tuple(np.array(cell["box"]) + (6, 8, -6, -8)), "table_cell", None, 9)
        else:
            add_text_token(tokens, 2, "Service ledger — continued", (55, 55, 410, 84), "heading", font_size=18)
            regions.append(RegionRecord(f"{doc_id}:p2:heading", 2, "heading", (45, 40, 755, 100), 0, text="Service ledger continued"))
            table = make_table(2, f"{doc_id}:ledger", 4, True)
            all_tables.append(table)
            regions.append(RegionRecord(f"{doc_id}:p2:table", 2, "table", table["box"], 1, text="service ledger continued"))
            for cell in table["cells"]:
                if cell["text"]:
                    add_text_token(tokens, 2, cell["text"], tuple(np.array(cell["box"]) + (6, 8, -6, -8)), "table_cell", None, 9)
            for i, (label, checked) in enumerate((("approved", True), ("rejected", False), ("requires review", doc_type != "invoice"))):
                y = 430 + i * 46
                add_text_token(tokens, 2, label, (105, y, 280, y + 24), "checkbox_label", label.replace(" ", "_"), 11)
                regions.append(RegionRecord(f"{doc_id}:p2:checkbox:{label.replace(' ', '_')}", 2, "form_field", (65, y - 5, 285, y + 29), 10 + i, text=label))
            regions.append(RegionRecord(f"{doc_id}:p2:figure", 2, "figure", (420, 430, 720, 680), 20, text="equipment figure"))
            add_text_token(tokens, 2, "Figure 1 — inspected assembly", (430, 695, 710, 720), "caption", font_size=10)
            regions.append(RegionRecord(f"{doc_id}:p2:caption", 2, "caption", (420, 685, 730, 730), 21, text="Figure 1 — inspected assembly"))
        add_text_token(tokens, page_number, "CONFIDENTIAL — controlled copy", (235, 948, 565, 972), "footer", font_size=8)
        regions.append(RegionRecord(f"{doc_id}:p{page_number}:footer", page_number, "footer", (40, 935, 760, 985), 99, text="CONFIDENTIAL — controlled copy"))
        page = {"page": page_number, "width": PAGE_WIDTH, "height": PAGE_HEIGHT, "dpi": 150,
                "tokens": tokens, "regions": [asdict(r) for r in regions]}
        pages.append(page)
        all_regions.extend(page["regions"])
        all_tokens.extend(tokens)
    logical_source = {"document_id": doc_id, "template": template, "document_type": doc_type, "pages": pages,
                      "tables": all_tables, "field_truth": values}
    logical_source["document_sha256"] = canonical_hash(logical_source)
    logical_source["split"] = SOURCE_CONTRACT[template]
    return logical_source


def render_page(document: dict, page_number: int):
    page = document["pages"][page_number - 1]
    style = TEMPLATE_STYLE[document["template"]]
    image = Image.new("RGB", (page["width"], page["height"]), style["paper"])
    draw = ImageDraw.Draw(image)
    draw.rectangle((25, 25, PAGE_WIDTH - 25, PAGE_HEIGHT - 25), outline=style["accent"], width=3)
    for region in page["regions"]:
        if region["kind"] == "table":
            draw.rectangle(region["box"], outline=(75, 84, 94), width=2)
        elif region["kind"] == "figure":
            draw.rounded_rectangle(region["box"], radius=12, fill=(222, 227, 232), outline=style["accent"], width=2)
            x1, y1, x2, y2 = region["box"]
            draw.ellipse((x1 + 70, y1 + 55, x2 - 70, y2 - 55), outline=(78, 86, 95), width=5)
        elif "checkbox" in region["region_id"]:
            x1, y1, _, _ = region["box"]
            draw.rectangle((x1 + 5, y1 + 7, x1 + 25, y1 + 27), outline=(40, 45, 50), width=2)
            if "approved" in region["region_id"] or ("requires_review" in region["region_id"] and document["document_type"] != "invoice"):
                draw.line((x1 + 8, y1 + 18, x1 + 14, y1 + 24, x1 + 24, y1 + 9), fill=style["accent"], width=3)
    for token in page["tokens"]:
        draw.text((token["box"][0], token["box"][1]), token["text"], fill=(26, 32, 39))
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return image, hashlib.sha256(buffer.getvalue()).hexdigest()


In [ ]:
corpus = [make_document(template, doc_type, serial)
          for template in TEMPLATE_STYLE
          for serial, doc_type in enumerate(DOC_FIELDS, start=1)]
corpus_index = pd.DataFrame([{k: doc[k] for k in ("document_id", "template", "document_type", "split", "document_sha256")} for doc in corpus])

# Source isolation is checked at document, page, token, and table level.
assert set(corpus_index.groupby("template")["split"].nunique()) == {1}
assert set(corpus_index.groupby("document_id")["split"].nunique()) == {1}
assert corpus_index.query("template == 'Template C'")["split"].eq("test_reporting_only").all()
for doc in corpus:
    assert all(validate_xyxy(region["box"], PAGE_WIDTH, PAGE_HEIGHT) for page in doc["pages"] for region in page["regions"])
    assert all(validate_xyxy(token["box"], PAGE_WIDTH, PAGE_HEIGHT) for page in doc["pages"] for token in page["tokens"])

fig, axes = plt.subplots(3, 2, figsize=(10, 15))
for row, template in enumerate(TEMPLATE_STYLE):
    doc = next(d for d in corpus if d["template"] == template and d["document_type"] == "invoice")
    for col, page in enumerate((1, 2)):
        image, page_hash = render_page(doc, page)
        axes[row, col].imshow(image)
        axes[row, col].set_title(f"{template} · invoice · page {page}\nrender {page_hash[:10]}")
        axes[row, col].axis("off")
plt.tight_layout()
corpus_index


## 5. OCR proxy, CER, and WER

OCR is two contracts: locate text and recognize it. `local_ocr_proxy` starts from the synthetic text layer, then injects declared recognition/detection errors. It is **not an OCR benchmark** and its hidden truth never appears in a production adapter interface.


In [ ]:
def levenshtein(reference: list[str], hypothesis: list[str]) -> tuple[int, int, int]:
    # Each DP entry stores total edits and (substitutions, deletions, insertions).
    dp = [[(0, 0, 0, 0) for _ in range(len(hypothesis) + 1)] for _ in range(len(reference) + 1)]
    for i in range(1, len(reference) + 1):
        dp[i][0] = (i, 0, i, 0)
    for j in range(1, len(hypothesis) + 1):
        dp[0][j] = (j, 0, 0, j)
    for i in range(1, len(reference) + 1):
        for j in range(1, len(hypothesis) + 1):
            if reference[i - 1] == hypothesis[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                s = dp[i - 1][j - 1]; d = dp[i - 1][j]; ins = dp[i][j - 1]
                choices = [(s[0] + 1, s[1] + 1, s[2], s[3]),
                           (d[0] + 1, d[1], d[2] + 1, d[3]),
                           (ins[0] + 1, ins[1], ins[2], ins[3] + 1)]
                dp[i][j] = min(choices, key=lambda value: value[0])
    _, s, d, i = dp[-1][-1]
    return s, d, i


def character_error_rate(reference: str, hypothesis: str) -> float:
    ref, hyp = list(reference), list(hypothesis)
    s, d, i = levenshtein(ref, hyp)
    if not ref:
        return 0.0 if not hyp else 1.0
    return (s + d + i) / len(ref)


def word_error_rate(reference: str, hypothesis: str) -> float:
    ref, hyp = reference.split(), hypothesis.split()
    s, d, i = levenshtein(ref, hyp)
    if not ref:
        return 0.0 if not hyp else 1.0
    return (s + d + i) / len(ref)


assert character_error_rate("ABC", "ADC") == 1 / 3
assert word_error_rate("invoice total", "invoice subtotal") == 1 / 2
assert character_error_rate("", "") == 0 and word_error_rate("", "extra") == 1
{"CER_known": character_error_rate("TOTAL 1200", "T0TAL 1200"), "WER_known": word_error_rate("TOTAL 1200", "T0TAL 1200")}


In [ ]:
SUBSTITUTIONS = str.maketrans({"0": "O", "1": "l", "5": "S"})


def perturb_word(text: str, mode: str, index: int, template: str, role: str) -> tuple[str | None, float, str]:
    confidence = 0.97
    output, error = text, "none"
    source_shift = template == "Template C" and role == "value" and index % 2 == 0
    if mode == "rotation" and index % 5 == 0:
        return None, 0.31, "missing_text"
    if mode == "blur" and index % 4 == 0:
        output, confidence, error = text.translate(SUBSTITUTIONS), 0.62, "character_substitution"
    elif mode == "compression" and len(text) > 7 and index % 6 == 0:
        output, confidence, error = text.replace(" ", "", 1), 0.72, "word_merge"
    elif mode == "low_contrast" and index % 7 == 0:
        return None, 0.42, "missing_text"
    elif source_shift:
        output, confidence, error = text.translate(SUBSTITUTIONS), 0.91, "template_shift_substitution"
    if index == 9 and mode == "clean" and template == "Template B":
        output, confidence, error = text[:-1] + "?", 0.96, "overconfident_substitution"
    return output, confidence, error


def local_ocr_proxy(document: dict, perturbation="clean") -> list[dict]:
    output = []
    for page in document["pages"]:
        for index, token in enumerate(page["tokens"]):
            text, confidence, error = perturb_word(token["text"], perturbation, index, document["template"], token["role"])
            if text is None:
                continue
            output.append({**token, "text": text, "reference_text": token["text"], "confidence": confidence,
                           "error_type": error, "engine": LOCAL_OCR_LABEL, "foundation_model": False,
                           "document_id": document["document_id"], "template": document["template"], "split": document["split"]})
    return output


def ocr_metrics(document: dict, perturbation="clean") -> dict:
    predicted = {row["word_id"]: row for row in local_ocr_proxy(document, perturbation)}
    ref_tokens = [token for page in document["pages"] for token in page["tokens"]]
    reference = " ".join(token["text"] for token in ref_tokens)
    hypothesis = " ".join(predicted[token["word_id"]]["text"] for token in ref_tokens if token["word_id"] in predicted)
    return {"document_id": document["document_id"], "template": document["template"], "split": document["split"],
            "perturbation": perturbation, "cer": character_error_rate(reference, hypothesis),
            "wer": word_error_rate(reference, hypothesis), "retained_word_rate": len(predicted) / len(ref_tokens)}


ocr_clean_results = pd.DataFrame([ocr_metrics(doc) for doc in corpus])
ocr_clean_results.groupby(["split", "template"])[["cer", "wer", "retained_word_rate"]].mean()


## 6. OCR confidence is not correctness

The next table compares engine telemetry with observed correctness. One deliberately overconfident substitution demonstrates why a confidence threshold cannot be selected without source-aware calibration.


In [ ]:
development_ocr = [row for doc in corpus if doc["template"] == "Template B" for row in local_ocr_proxy(doc)]
reliability = pd.DataFrame([{
    "confidence_bin": pd.cut([row["confidence"]], bins=[0, .6, .8, .95, 1.0], labels=["≤.60", ".60–.80", ".80–.95", ">.95"])[0],
    "correct": row["text"] == row["reference_text"], "font_size": row["font_size"], "role": row["role"],
    "confidence": row["confidence"], "error_type": row["error_type"],
} for row in development_ocr])
confidence_reliability = reliability.groupby("confidence_bin", observed=False).agg(samples=("correct", "size"), accuracy=("correct", "mean"), mean_confidence=("confidence", "mean")).reset_index()
assert ((reliability["confidence"] > .95) & (~reliability["correct"])).any()
confidence_reliability


## 7. Layout and reading-order contracts

`local_layout_proxy` perturbs known regions to expose evaluation mechanics. Region classification and overlap are measured independently. Reading order is then evaluated as a graph problem rather than assumed from OCR output order.

![Common document layout region types.](assets/layout-taxonomy.svg)


In [ ]:
def box_iou(first, second) -> float:
    ax1, ay1, ax2, ay2 = map(float, first); bx1, by1, bx2, by2 = map(float, second)
    iw, ih = max(0, min(ax2, bx2) - max(ax1, bx1)), max(0, min(ay2, by2) - max(ay1, by1))
    intersection = iw * ih
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - intersection
    return intersection / union if union else 0.0


def local_layout_proxy(document: dict, perturbation="clean") -> list[dict]:
    predictions = []
    for page in document["pages"]:
        for index, region in enumerate(page["regions"]):
            predicted = deepcopy(region)
            predicted["engine"] = LOCAL_LAYOUT_LABEL
            predicted["foundation_model"] = False
            if document["template"] == "Template C" and region["kind"] == "caption":
                predicted["kind"] = "paragraph"
            if perturbation == "rotation" and index % 4 == 0:
                x1, y1, x2, y2 = predicted["box"]
                predicted["box"] = (min(x1 + 16, x2 - 1), y1, x2, y2)
            if perturbation == "cropping" and region["kind"] == "footer":
                continue
            predictions.append(predicted)
    return predictions


def layout_metrics(document: dict, perturbation="clean") -> dict:
    truth = {r["region_id"]: r for p in document["pages"] for r in p["regions"]}
    pred = {r["region_id"]: r for r in local_layout_proxy(document, perturbation)}
    shared = sorted(set(truth) & set(pred))
    return {"region_recall": len(shared) / len(truth),
            "layout_class_accuracy": np.mean([truth[k]["kind"] == pred[k]["kind"] for k in shared]) if shared else 0,
            "mean_region_iou": np.mean([box_iou(truth[k]["box"], pred[k]["box"]) for k in shared]) if shared else 0}


layout_summary = pd.DataFrame([{**{"template": d["template"], "document_id": d["document_id"]}, **layout_metrics(d)} for d in corpus])
layout_summary.groupby("template")[["region_recall", "layout_class_accuracy", "mean_region_iou"]].mean()


In [ ]:
def naive_yx_order(regions: list[dict]) -> list[str]:
    return [r["region_id"] for r in sorted(regions, key=lambda r: (r["box"][1], r["box"][0]))]


def column_aware_order(regions: list[dict]) -> list[str]:
    # Controlled-layout primitive: header, columns left-to-right, then footer.
    header = [r for r in regions if r["kind"] in {"title", "heading", "header"}]
    footer = [r for r in regions if r["kind"] == "footer"]
    body = [r for r in regions if r not in header and r not in footer]
    ordered = sorted(header, key=lambda r: r["box"][1])
    for column in sorted({r.get("column", 0) for r in body}):
        ordered.extend(sorted((r for r in body if r.get("column", 0) == column), key=lambda r: (r["box"][1], r["box"][0])))
    ordered.extend(sorted(footer, key=lambda r: r["box"][1]))
    return [r["region_id"] for r in ordered]


def pairwise_order_accuracy(reference: list[str], candidate: list[str]) -> float:
    reference_position = {value: i for i, value in enumerate(reference)}
    candidate_position = {value: i for i, value in enumerate(candidate)}
    shared = [value for value in reference if value in candidate_position]
    pairs = [(shared[i], shared[j]) for i in range(len(shared)) for j in range(i + 1, len(shared))]
    return np.mean([(reference_position[a] < reference_position[b]) == (candidate_position[a] < candidate_position[b]) for a, b in pairs]) if pairs else 1.0


reading_regions = [
    {"region_id": "title", "kind": "title", "box": (40, 30, 760, 80), "column": 0},
    *[{"region_id": f"A{i}", "kind": "paragraph", "box": (50, 100 + i * 90, 350, 160 + i * 90), "column": 0} for i in range(3)],
    *[{"region_id": f"B{i}", "kind": "paragraph", "box": (430, 100 + i * 90, 750, 160 + i * 90), "column": 1} for i in range(3)],
    {"region_id": "footer", "kind": "footer", "box": (40, 900, 760, 950), "column": 0},
]
intended = ["title", "A0", "A1", "A2", "B0", "B1", "B2", "footer"]
naive_order = naive_yx_order(reading_regions)
graph_order = column_aware_order(reading_regions)
reading_order_metrics = pd.DataFrame([
    {"method": "naive y/x", "order": naive_order, "pairwise_accuracy": pairwise_order_accuracy(intended, naive_order)},
    {"method": "column-aware graph", "order": graph_order, "pairwise_accuracy": pairwise_order_accuracy(intended, graph_order)},
])
assert graph_order == intended and reading_order_metrics.iloc[1]["pairwise_accuracy"] == 1
reading_order_metrics


## 8. Tables: detection, structure, text, and spans

Flattening makes the content look convenient while silently deleting cell geometry, merged-header spans, blank cells, and page continuation.

![Rows, columns, cells, headers, and merged spans remain explicit.](assets/table-structure.svg)


In [ ]:
def validate_table_schema(table: dict) -> list[str]:
    errors = []
    ids = set()
    for cell in table["cells"]:
        if cell["cell_id"] in ids:
            errors.append("duplicate_cell_id")
        ids.add(cell["cell_id"])
        if cell["row_end"] < cell["row_start"] or cell["col_end"] < cell["col_start"]:
            errors.append("invalid_span")
        if not validate_xyxy(cell["box"], PAGE_WIDTH, PAGE_HEIGHT):
            errors.append("invalid_cell_box")
    return errors


def flatten_table(table: dict) -> str:
    by_row = {}
    for cell in table["cells"]:
        by_row.setdefault(cell["row_start"], []).append(cell)
    return "\n".join(" | ".join(cell["text"] for cell in sorted(row, key=lambda c: c["col_start"])) for _, row in sorted(by_row.items()))


example_table = next(doc for doc in corpus if doc["template"] == "Template A" and doc["document_type"] == "invoice")["tables"][0]
assert not validate_table_schema(example_table)
assert any(cell["col_end"] - cell["col_start"] == 2 for cell in example_table["cells"])
table_flattening_loss = {
    "flattened_text": flatten_table(example_table),
    "merged_span_count_before": sum(c["col_end"] > c["col_start"] for c in example_table["cells"]),
    "merged_span_count_after_plain_text": 0,
    "geometry_preserved_after_plain_text": False,
}
table_flattening_loss


In [ ]:
def table_structure_metrics(truth: dict, prediction: dict) -> dict:
    truth_cells = {c["cell_id"]: c for c in truth["cells"]}
    pred_cells = {c["cell_id"]: c for c in prediction["cells"]}
    shared = sorted(set(truth_cells) & set(pred_cells))
    exact_structure = [
        (truth_cells[k]["row_start"], truth_cells[k]["row_end"], truth_cells[k]["col_start"], truth_cells[k]["col_end"])
        == (pred_cells[k]["row_start"], pred_cells[k]["row_end"], pred_cells[k]["col_start"], pred_cells[k]["col_end"])
        for k in shared
    ]
    return {
        "cell_recall": len(shared) / len(truth_cells),
        "row_column_assignment_accuracy": float(np.mean(exact_structure)) if shared else 0,
        "merged_cell_correctness": float(np.mean([exact_structure[i] for i, k in enumerate(shared) if truth_cells[k]["col_end"] > truth_cells[k]["col_start"]])) if any(truth_cells[k]["col_end"] > truth_cells[k]["col_start"] for k in shared) else 1.0,
        "exact_cell_text": float(np.mean([truth_cells[k]["text"] == pred_cells[k]["text"] for k in shared])) if shared else 0,
    }


table_prediction = deepcopy(example_table)
table_metrics_clean = table_structure_metrics(example_table, table_prediction)
broken_span_prediction = deepcopy(example_table)
broken_span_prediction["cells"][0]["col_end"] = 0
table_metrics_broken_span = table_structure_metrics(example_table, broken_span_prediction)
assert table_metrics_clean["merged_cell_correctness"] == 1
assert table_metrics_broken_span["merged_cell_correctness"] == 0
pd.DataFrame([{"case": "clean", **table_metrics_clean}, {"case": "merged-cell failure", **table_metrics_broken_span}])


## 9. Key/value binding: proximity can be confidently wrong

The baseline sees only distance. The improved extractor recognizes an approved key vocabulary, checks value shape, then uses geometry. Both remain deterministic and inspectable.


In [ ]:
KEY_ALIASES = {
    "invoice number": "invoice_number", "purchase order number": "purchase_order", "invoice date": "date", "supplier": "supplier",
    "report id": "report_id", "inspection date": "date", "site": "site", "quality score": "score",
    "work order": "work_order", "service date": "date", "asset id": "asset", "status": "status",
}


def center(box):
    x1, y1, x2, y2 = map(float, box)
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def euclidean_boxes(first, second):
    return math.dist(center(first), center(second))


def proximity_bind(ocr_rows: list[dict]) -> dict[str, dict]:
    keys = [r for r in ocr_rows if r["role"] == "key"]
    values = [r for r in ocr_rows if r["role"] == "value"]
    bindings = {}
    for key in keys:
        candidate = min(values, key=lambda value: euclidean_boxes(key["box"], value["box"]))
        bindings[KEY_ALIASES.get(key["text"].lower(), key["text"].lower())] = {"key": key, "value": candidate, "method": "nearest_geometry_v1"}
    return bindings


def value_shape_score(field_name: str, value: str) -> float:
    patterns = {
        "invoice_number": r"^INV-\d+$", "purchase_order": r"^PO-\d+$", "report_id": r"^RPT-\d+$",
        "work_order": r"^WO-\d+$", "date": r"(?:\d{1,4}[-/.]|[A-Za-z]{3,})", "score": r"%$",
        "asset": r"^[A-Z]+-\d+$", "supplier": r"^[A-Za-z][A-Za-z ]+$", "site": r"^Plant \d+$",
        "status": r"^(Requires review|Approved|Rejected)$",
    }
    return 1.0 if field_name not in patterns or re.search(patterns[field_name], value) else 0.0


def semantic_geometry_bind(ocr_rows: list[dict]) -> dict[str, dict]:
    keys = [r for r in ocr_rows if r["role"] == "key" and r["text"].lower() in KEY_ALIASES]
    values = [r for r in ocr_rows if r["role"] == "value"]
    bindings = {}
    for key in keys:
        field_name = KEY_ALIASES[key["text"].lower()]
        ranked = sorted(values, key=lambda value: (-value_shape_score(field_name, value["text"]), euclidean_boxes(key["box"], value["box"])))
        best = ranked[0]
        bindings[field_name] = {"key": key, "value": best, "method": "semantic_key_plus_geometry_v1",
                                "shape_score": value_shape_score(field_name, best["text"])}
    return bindings


def binding_accuracy(document: dict, method) -> float:
    bindings = method(local_ocr_proxy(document))
    comparisons = [binding["value"]["reference_text"] == document["field_truth"][name]
                   for name, binding in bindings.items() if name in document["field_truth"]]
    return float(np.mean(comparisons)) if comparisons else 0.0


binding_results = pd.DataFrame([{
    "template": doc["template"], "document_type": doc["document_type"],
    "proximity_accuracy": binding_accuracy(doc, proximity_bind),
    "semantic_geometry_accuracy": binding_accuracy(doc, semantic_geometry_bind),
} for doc in corpus])
binding_results.groupby("template")[["proximity_accuracy", "semantic_geometry_accuracy"]].mean()


In [ ]:
# A controlled collision: the closest string to Invoice Number is a PO identifier.
proximity_failure_rows = [
    {"word_id": "k-inv", "text": "Invoice Number", "reference_text": "Invoice Number", "box": (50, 50, 200, 75), "role": "key"},
    {"word_id": "k-po", "text": "Purchase Order Number", "reference_text": "Purchase Order Number", "box": (50, 100, 230, 125), "role": "key"},
    {"word_id": "v-po", "text": "PO-871", "reference_text": "PO-871", "box": (220, 50, 300, 75), "role": "value"},
    {"word_id": "v-inv", "text": "INV-1041", "reference_text": "INV-1041", "box": (220, 100, 320, 125), "role": "value"},
]
nearest = proximity_bind(proximity_failure_rows)
improved = semantic_geometry_bind(proximity_failure_rows)
proximity_failure = {
    "nearest_invoice_value": nearest["invoice_number"]["value"]["text"],
    "semantic_geometry_invoice_value": improved["invoice_number"]["value"]["text"],
}
assert proximity_failure["nearest_invoice_value"] == "PO-871"
assert proximity_failure["semantic_geometry_invoice_value"] == "INV-1041"
proximity_failure


## 10. Normalization is a versioned transformation

Raw evidence is immutable. A normalizer records its version, assumptions, output, and replay status. Ambiguous dates remain uncertain until a locale or contextual rule is supplied.


In [ ]:
@dataclass(frozen=True)
class TransformationRecord:
    name: str
    version: str
    input: str
    output: Any
    assumptions: dict[str, Any]


def normalize_currency(raw: str, currency="CAD") -> TransformationRecord:
    value = float(re.sub(r"[^0-9.\-]", "", raw.replace(",", "")))
    return TransformationRecord("currency_parser", "1.0.0", raw, value, {"currency": currency})


def normalize_percentage(raw: str) -> TransformationRecord:
    value = float(raw.strip().removesuffix("%")) / 100
    return TransformationRecord("percentage_parser", "1.0.0", raw, value, {"scale": "fraction"})


def normalize_identifier(raw: str) -> TransformationRecord:
    value = re.sub(r"\s+", "", raw).upper()
    return TransformationRecord("identifier_normalizer", "1.0.0", raw, value, {"case": "upper", "whitespace": "removed"})


def normalize_date(raw: str, locale: str | None = None) -> tuple[str, TransformationRecord | None]:
    if re.fullmatch(r"\d{2}/\d{2}/\d{2,4}", raw):
        first, second, year = raw.split("/")
        if int(first) <= 12 and int(second) <= 12 and locale is None:
            return "uncertain", None
        fmt = "%m/%d/%y" if locale == "en_US" else "%d/%m/%y"
        value = datetime.strptime(raw, fmt).date().isoformat()
        return "verified", TransformationRecord("date_normalizer", "2.0.0", raw, value, {"locale": locale})
    for fmt in ("%Y-%m-%d", "%b. %d, %Y", "%b %d, %Y"):
        try:
            value = datetime.strptime(raw, fmt).date().isoformat()
            return "verified", TransformationRecord("date_normalizer", "2.0.0", raw, value, {"format": fmt})
        except ValueError:
            pass
    return "uncertain", None


assert normalize_currency("$1,240.00").output == 1240.0
assert normalize_percentage("97.5%").output == 0.975
assert normalize_identifier(" inv-1042 ").output == "INV-1042"
assert normalize_date("03/04/2026")[0] == "uncertain"
assert normalize_date("09/07/26", "en_CA")[1].output == "2026-07-09"
assert normalize_date("09/07/26", "en_US")[1].output == "2026-09-07"
ambiguous_date_policy = pd.DataFrame([
    {"raw": "03/04/2026", "locale": None, "state": normalize_date("03/04/2026")[0], "normalized": None},
    {"raw": "03/04/26", "locale": "en_CA", "state": "verified", "normalized": normalize_date("03/04/26", "en_CA")[1].output},
    {"raw": "03/04/26", "locale": "en_US", "state": "verified", "normalized": normalize_date("03/04/26", "en_US")[1].output},
])
ambiguous_date_policy


## 11. Visual controls and cross-page continuation

A checked box is a visual state, not text. A signature region can establish presence, not authenticity. Cross-page table merging requires adjacent pages, repeated headers, compatible columns, and retained cell-level page provenance.


In [ ]:
def checkbox_proxy(document: dict) -> list[dict]:
    states = {"approved": True, "rejected": False, "requires_review": document["document_type"] != "invoice"}
    regions = [r for r in document["pages"][1]["regions"] if "checkbox" in r["region_id"]]
    return [{"field": name, "checked": states[name], "document_id": document["document_id"], "page": 2,
             "box": next(r["box"] for r in regions if name in r["region_id"]), "engine": "local_checkbox_proxy",
             "signature_authenticity": "not_evaluated"} for name in states]


def detect_table_continuation(first: dict, second: dict) -> dict:
    first_headers = [c["text"] for c in first["cells"] if c["role"] == "column_header"]
    second_headers = [c["text"] for c in second["cells"] if c["role"] == "column_header"]
    aligned = first["columns"] == second["columns"] and np.allclose(
        sorted({c["box"][0] for c in first["cells"] if c["role"] == "column_header"}),
        sorted({c["box"][0] for c in second["cells"] if c["role"] == "column_header"}), atol=3)
    adjacent = second["page"] == first["page"] + 1
    score = np.mean([first_headers == second_headers, aligned, adjacent, second["continued"]])
    return {"continuation": bool(score >= .75), "confidence": float(score), "signals": {
        "repeated_headers": first_headers == second_headers, "column_alignment": bool(aligned), "adjacent_pages": adjacent}}


def detect_repeated_boilerplate(document: dict) -> list[dict]:
    occurrences = {}
    for page in document["pages"]:
        for token in page["tokens"]:
            key = (token["text"].strip().lower(), round(token["box"][1] / PAGE_HEIGHT, 1))
            occurrences.setdefault(key, []).append((page["page"], token["word_id"]))
    return [{"text": text, "vertical_band": band, "occurrences": rows, "suppressed_from_body": True,
             "retained_for_provenance": True} for (text, band), rows in occurrences.items() if len(rows) > 1]


def bind_figures_to_captions(document: dict) -> list[dict]:
    page = document["pages"][1]
    figures = [r for r in page["regions"] if r["kind"] == "figure"]
    captions = [r for r in page["regions"] if r["kind"] == "caption"]
    return [{"figure_region_id": figure["region_id"],
             "caption_region_id": min(captions, key=lambda caption: euclidean_boxes(figure["box"], caption["box"]))["region_id"],
             "binding_method": "nearest_below_geometry_v1", "ambiguity_review": len(captions) != 1}
            for figure in figures]


invoice_a = next(doc for doc in corpus if doc["template"] == "Template A" and doc["document_type"] == "invoice")
continuation = detect_table_continuation(invoice_a["tables"][0], invoice_a["tables"][1])
merged_cells = invoice_a["tables"][0]["cells"] + invoice_a["tables"][1]["cells"]
assert continuation["continuation"]
assert {cell["page"] for cell in merged_cells} == {1, 2}
boilerplate = detect_repeated_boilerplate(invoice_a)
assert any("confidential" in row["text"] for row in boilerplate)
{"checkboxes": checkbox_proxy(invoice_a), "continuation": continuation,
 "merged_cell_pages": sorted({cell["page"] for cell in merged_cells}),
 "repeated_boilerplate": boilerplate, "figure_caption_bindings": bind_figures_to_captions(invoice_a)}


## 12. Structured document and evidence graph

We now build the output schema. Every field carries raw and normalized values plus a path to word or cell, region, page render, and original document hash.

![A field traces through a span or cell, region, page, and original document.](assets/document-evidence-graph.svg)


In [ ]:
def page_region_for_box(document: dict, page_number: int, box) -> str | None:
    regions = document["pages"][page_number - 1]["regions"]
    candidates = [(box_iou(region["box"], box), region["region_id"]) for region in regions]
    return max(candidates)[1] if candidates and max(candidates)[0] > 0 else None


def normalization_for(field_name: str, raw: str, locale="en_US"):
    if field_name == "invoice_total":
        return "verified", normalize_currency(raw)
    if field_name == "date":
        return normalize_date(raw, locale)
    if field_name == "score":
        return "verified", normalize_percentage(raw)
    if field_name in {"invoice_number", "purchase_order", "report_id", "work_order", "asset"}:
        return "verified", normalize_identifier(raw)
    return "verified", TransformationRecord("identity", "1.0.0", raw, raw, {})


def build_structured_document(document: dict, locale="en_US") -> dict:
    page_hashes = {page: render_page(document, page)[1] for page in (1, 2)}
    ocr_rows = local_ocr_proxy(document)
    bindings = semantic_geometry_bind(ocr_rows)
    fields = {}
    for field_name, binding in bindings.items():
        raw = binding["value"]["text"]
        try:
            state, transform = normalization_for(field_name, raw, locale)
        except (TypeError, ValueError):
            state, transform = "uncertain", None
        page = binding["value"]["page"]
        box = binding["value"]["box"]
        fields[field_name] = {
            "raw": raw, "normalized": transform.output if transform else None, "state": state,
            "evidence": {"document_id": document["document_id"], "document_sha256": document["document_sha256"],
                         "page": page, "page_render_sha256": page_hashes[page],
                         "region_id": page_region_for_box(document, page, box), "word_ids": [binding["value"]["word_id"]],
                         "cell_id": None, "box": box, "source_text": raw, "engine": LOCAL_OCR_LABEL},
            "transformation": asdict(transform) if transform else None,
            "binding_method": binding["method"],
        }
    if document["document_type"] == "invoice":
        total_cell = next(cell for table in document["tables"] for cell in table["cells"] if cell["text"] == "$1,240.00")
        state, transform = normalization_for("invoice_total", total_cell["text"], locale)
        fields["invoice_total"] = {
            "raw": total_cell["text"], "normalized": transform.output, "state": state,
            "evidence": {"document_id": document["document_id"], "document_sha256": document["document_sha256"],
                         "page": total_cell["page"], "page_render_sha256": page_hashes[total_cell["page"]],
                         "region_id": f"{document['document_id']}:p{total_cell['page']}:table", "word_ids": [],
                         "cell_id": total_cell["cell_id"], "box": total_cell["box"], "source_text": total_cell["text"],
                         "engine": "local_table_proxy"},
            "transformation": asdict(transform), "binding_method": "table_cell_binding_v1",
        }
    return {
        "document_id": document["document_id"], "document_sha256": document["document_sha256"],
        "document_type": document["document_type"], "template": document["template"], "split": document["split"],
        "pages": [{"page": p["page"], "width": p["width"], "height": p["height"], "dpi": p["dpi"],
                   "page_render_sha256": page_hashes[p["page"]]} for p in document["pages"]],
        "fields": fields, "tables": document["tables"], "figures": [r for p in document["pages"] for r in p["regions"] if r["kind"] in {"figure", "caption"}],
        "checkboxes": checkbox_proxy(document), "provenance_engine": "deterministic_teaching_pipeline_v1",
    }


structured_invoice = build_structured_document(invoice_a)
assert structured_invoice["fields"]["invoice_total"]["evidence"]["page"] == 2
assert structured_invoice["fields"]["invoice_total"]["evidence"]["cell_id"] is not None
pd.DataFrame([{"field": name, "raw": value["raw"], "normalized": value["normalized"], "state": value["state"],
               "page": value["evidence"]["page"], "region": value["evidence"]["region_id"], "cell": value["evidence"]["cell_id"]}
              for name, value in structured_invoice["fields"].items()])


## 13. Deterministic evidence verification and failure injection

Verification checks referential integrity and replay. It does not accept a value because it is plausible. The lab injects wrong-page, wrong-box, wrong-cell, unsupported-value, and bad-normalization failures and attributes the earliest violated boundary.


In [ ]:
def replay_transformation(record: dict | None):
    if record is None:
        return None
    name, raw, assumptions = record["name"], record["input"], record["assumptions"]
    if name == "currency_parser": return normalize_currency(raw, assumptions["currency"]).output
    if name == "percentage_parser": return normalize_percentage(raw).output
    if name == "identifier_normalizer": return normalize_identifier(raw).output
    if name == "date_normalizer": return normalize_date(raw, assumptions.get("locale"))[1].output
    if name == "identity": return raw
    raise KeyError(f"unknown transformation {name}")


def verify_field(document: dict, field_name: str, field_record: dict) -> dict:
    evidence = field_record.get("evidence", {})
    pages = {page["page"]: page for page in document["pages"]}
    checks = {
        "document_hash_matches": evidence.get("document_sha256") == document["document_sha256"],
        "page_exists": evidence.get("page") in pages,
        "box_valid": False,
        "region_exists": False,
        "source_exists": False,
        "normalization_replays": False,
    }
    if checks["page_exists"]:
        page = pages[evidence["page"]]
        checks["box_valid"] = validate_xyxy(evidence.get("box", (0, 0, 0, 0)), page["width"], page["height"])
        checks["region_exists"] = evidence.get("region_id") in {r["region_id"] for r in page["regions"]}
        word_ids = set(evidence.get("word_ids", []))
        word_match = any(t["word_id"] in word_ids and t["text"] == evidence.get("source_text") for t in page["tokens"])
        cell_match = any(c["cell_id"] == evidence.get("cell_id") and c["text"] == evidence.get("source_text") for t in document["tables"] for c in t["cells"])
        checks["source_exists"] = bool(word_match or cell_match)
    try:
        checks["normalization_replays"] = replay_transformation(field_record.get("transformation")) == field_record.get("normalized")
    except Exception:
        checks["normalization_replays"] = False
    failure_order = ["document_hash_matches", "page_exists", "box_valid", "region_exists", "source_exists", "normalization_replays"]
    earliest = next((name for name in failure_order if not checks[name]), None)
    state = "verified" if earliest is None else ("unsupported" if earliest == "source_exists" else "uncertain")
    return {"field": field_name, "state": state, "checks": checks, "earliest_failure": earliest}


clean_verification = {name: verify_field(invoice_a, name, record) for name, record in structured_invoice["fields"].items()}
assert all(result["state"] == "verified" for result in clean_verification.values())
clean_verification


In [ ]:
def inject_provenance_failure(structured: dict, field_name: str, failure: str):
    broken = deepcopy(structured["fields"][field_name])
    if failure == "wrong_page": broken["evidence"]["page"] = 99
    elif failure == "wrong_box": broken["evidence"]["box"] = (-5, 10, 4, 20)
    elif failure == "wrong_cell": broken["evidence"]["cell_id"] = "missing-cell"
    elif failure == "unsupported_value":
        broken["raw"] = broken["evidence"]["source_text"] = "$1,450.00"
        broken["normalized"] = 1450.0
        broken["transformation"] = asdict(normalize_currency("$1,450.00"))
    elif failure == "bad_normalization": broken["normalized"] = 1450.0
    else: raise ValueError(failure)
    return broken


failure_rows = []
for failure in ("wrong_page", "wrong_box", "wrong_cell", "unsupported_value", "bad_normalization"):
    broken = inject_provenance_failure(structured_invoice, "invoice_total", failure)
    result = verify_field(invoice_a, "invoice_total", broken)
    failure_rows.append({"injection": failure, "state": result["state"], "earliest_failure": result["earliest_failure"], **result["checks"]})
provenance_failure_attribution = pd.DataFrame(failure_rows)
assert provenance_failure_attribution["state"].ne("verified").all()
assert provenance_failure_attribution.query("injection == 'unsupported_value'").iloc[0]["state"] == "unsupported"
provenance_failure_attribution


## 14. Document classification before extraction

A document type selects a schema. Misclassification therefore cascades into missing and unsupported fields. This transparent proxy classifies from the title span; production systems must evaluate page-, section-, and document-level evidence.


In [ ]:
def classify_document(ocr_rows: list[dict]) -> dict:
    title = " ".join(row["text"].lower() for row in ocr_rows if row["role"] == "title")
    for doc_type in DOC_FIELDS:
        if doc_type.replace("_", " ") in title:
            return {"document_type": doc_type, "state": "verified", "evidence_word_ids": [r["word_id"] for r in ocr_rows if r["role"] == "title"]}
    return {"document_type": None, "state": "uncertain", "evidence_word_ids": []}


classification_results = pd.DataFrame([{
    "document_id": doc["document_id"], "truth": doc["document_type"],
    **classify_document(local_ocr_proxy(doc))
} for doc in corpus])
classification_accuracy = float((classification_results["truth"] == classification_results["document_type"]).mean())

wrong_schema_example = deepcopy(classification_results.iloc[0].to_dict())
wrong_schema_example["document_type"] = "maintenance_form"
wrong_schema_example["cascading_risk"] = "invoice fields omitted; maintenance fields requested without support"
classification_accuracy, wrong_schema_example


## 15. Template shift: development versus untouched reporting

Template B may inform policy. Template C is evaluated once using frozen code. The local proxies intentionally contain a mild C-specific OCR and caption-layout shift. This validates the evaluation method, not real vendor performance.


In [ ]:
def field_exact_match(document: dict, structured: dict) -> tuple[float, float]:
    truth = document["field_truth"]
    field_names = sorted(truth)
    raw = np.mean([name in structured["fields"] and structured["fields"][name]["raw"] == truth[name] for name in field_names]) if field_names else 0
    normalized = []
    for name in field_names:
        state, expected = normalization_for(name, truth[name], "en_US")
        normalized.append(name in structured["fields"] and state == structured["fields"][name]["state"] and
                          (expected.output if expected else None) == structured["fields"][name]["normalized"])
    return float(raw), float(np.mean(normalized)) if normalized else 0


def provenance_accuracy(document: dict, structured: dict) -> float:
    results = [name in structured["fields"] and verify_field(document, name, structured["fields"][name])["state"] == "verified"
               for name in sorted(document["field_truth"])]
    return float(np.mean(results)) if results else 0


def evaluate_document(document: dict) -> dict:
    structured = build_structured_document(document)
    raw_em, norm_acc = field_exact_match(document, structured)
    ocr = ocr_metrics(document)
    layout = layout_metrics(document)
    table_scores = [table_structure_metrics(table, deepcopy(table))["row_column_assignment_accuracy"] for table in document["tables"]]
    verification = [verify_field(document, name, value) for name, value in structured["fields"].items()]
    required = REQUIRED_FIELDS[document["document_type"]] if "REQUIRED_FIELDS" in globals() else set(document["field_truth"])
    nonverified = sum(result["state"] != "verified" for result in verification)
    missing_required = len(required - set(structured["fields"]))
    return {"document_id": document["document_id"], "template": document["template"], "split": document["split"],
            "document_classification_accuracy": classify_document(local_ocr_proxy(document))["document_type"] == document["document_type"],
            "ocr_character_accuracy": 1 - ocr["cer"], "layout_class_accuracy": layout["layout_class_accuracy"],
            "field_exact_match": raw_em, "normalized_value_accuracy": norm_acc,
            "reading_order_accuracy": float(reading_order_metrics.query("method == 'column-aware graph'")["pairwise_accuracy"].iloc[0]),
            "table_structure_accuracy": float(np.mean(table_scores)), "provenance_accuracy": provenance_accuracy(document, structured),
            "unsupported_extraction_rate": nonverified / max(len(verification), 1),
            "review_required": bool(nonverified or missing_required)}


frozen_evaluation_policy = {
    "selected_on": "Template B development only",
    "held_out_rule": "Template C reporting only; no threshold, parser, key alias, locale, or schema changes",
    "required_field_states_for_auto_accept": ["verified"],
}
evaluation_results = pd.DataFrame([evaluate_document(doc) for doc in corpus if doc["template"] in {"Template B", "Template C"}])
template_shift = evaluation_results.groupby(["split", "template"]).mean(numeric_only=True).reset_index()
assert evaluation_results.query("template == 'Template C'")["split"].eq("test_reporting_only").all()
template_shift


## 16. Perturbation tests report the failing stage

Rotation, blur, compression, low contrast, and cropping are applied after source assignment. OCR and layout degradation stay separate from downstream extraction so a final error is not misattributed.


In [ ]:
def perturb_render(image: Image.Image, mode: str) -> Image.Image:
    if mode == "rotation": return image.rotate(2.5, resample=Image.Resampling.BICUBIC, fillcolor="white")
    if mode == "blur": return image.filter(ImageFilter.GaussianBlur(1.4))
    if mode == "low_contrast": return ImageEnhance.Contrast(image).enhance(.45)
    if mode == "compression":
        buffer = io.BytesIO(); image.save(buffer, format="JPEG", quality=25); buffer.seek(0)
        return Image.open(buffer).convert("RGB")
    if mode == "cropping": return image.crop((0, 0, image.width, image.height - 55)).resize(image.size)
    return image


perturbation_rows = []
for document in (d for d in corpus if d["template"] in {"Template B", "Template C"}):
    base_image, _ = render_page(document, 1)
    for perturbation in ("clean", "rotation", "blur", "compression", "low_contrast", "cropping"):
        rendered = perturb_render(base_image, perturbation)
        ocr_mode = perturbation if perturbation in {"rotation", "blur", "compression", "low_contrast"} else "clean"
        layout_mode = perturbation if perturbation in {"rotation", "cropping"} else "clean"
        ocr = ocr_metrics(document, ocr_mode)
        layout = layout_metrics(document, layout_mode)
        perturbation_rows.append({"template": document["template"], "split": document["split"], "document_id": document["document_id"],
                                  "perturbation": perturbation, "render_megapixels": rendered.width * rendered.height / 1e6,
                                  "ocr_character_accuracy": 1 - ocr["cer"], "ocr_word_accuracy": 1 - ocr["wer"],
                                  "layout_class_accuracy": layout["layout_class_accuracy"], "layout_region_recall": layout["region_recall"]})
perturbation_results = pd.DataFrame(perturbation_rows)
perturbation_summary = perturbation_results.groupby(["split", "template", "perturbation"])[["ocr_character_accuracy", "ocr_word_accuracy", "layout_class_accuracy", "layout_region_recall"]].mean().reset_index()
perturbation_summary


## 17. Field-level review policy

A required field in `uncertain`, `missing`, `conflicting`, or `unsupported` state routes the document to review. A single document confidence score would hide which value and which evidence boundary failed.


In [ ]:
REQUIRED_FIELDS = {
    "invoice": {"invoice_number", "date", "supplier", "invoice_total"},
    "inspection_report": {"report_id", "date", "site", "score"},
    "maintenance_form": {"work_order", "date", "asset", "status"},
}


def review_decision(structured: dict) -> dict:
    required = REQUIRED_FIELDS[structured["document_type"]]
    missing = sorted(required - set(structured["fields"]))
    invalid = sorted(name for name in required & set(structured["fields"]) if structured["fields"][name]["state"] != "verified")
    review = bool(missing or invalid)
    return {"decision": "review_required" if review else "eligible_for_policy_evaluation",
            "missing_fields": missing, "nonverified_fields": invalid, "authorization": "none"}


def resolve_field_candidates(candidates: list[dict], policy: str | None = None) -> dict:
    supported = [candidate for candidate in candidates if candidate["state"] == "verified"]
    distinct = {candidate["normalized"] for candidate in supported}
    if len(distinct) > 1 and policy is None:
        return {"state": "conflicting", "selected": None, "candidates": supported}
    if len(distinct) > 1 and policy == "prefer_signed_summary":
        signed = [candidate for candidate in supported if candidate.get("signed_summary")]
        return {"state": "verified" if len(signed) == 1 else "conflicting", "selected": signed[0] if len(signed) == 1 else None,
                "candidates": supported, "policy": policy}
    return {"state": "verified" if supported else "missing", "selected": supported[0] if supported else None, "candidates": supported}


conflicting_value_example = resolve_field_candidates([
    {"normalized": 1200.0, "page": 1, "state": "verified", "signed_summary": False},
    {"normalized": 1250.0, "page": 2, "state": "verified", "signed_summary": True},
])
assert conflicting_value_example["state"] == "conflicting"


review_records = []
for document in corpus:
    structured = build_structured_document(document)
    for field_name, field_record in structured["fields"].items():
        field_record["state"] = verify_field(document, field_name, field_record)["state"]
    decision = review_decision(structured)
    review_records.append({"document_id": document["document_id"], "template": document["template"], **decision})
review_policy_results = pd.DataFrame(review_records)
review_rate_by_source = review_policy_results.assign(review=lambda frame: frame["decision"].eq("review_required")).groupby("template")["review"].mean()
review_rate_by_source


## 18. Governed optional adapters

Optional tools are disabled by default and run in isolated environments. Local proxy evidence and downloaded-model observations are never mixed. Model names alone are insufficient: record immutable revision, processor, artifact hashes, license, runtime, prompts/configuration, and input policy.


In [ ]:
OPTIONAL_TOOL_MANIFESTS = {
    "tesseract": {
        "enabled_env": "CV_ENABLE_TESSERACT", "version": "5.5.3",
        "source_revision": "6951ffe10ce031374bcd04fe400811da1e7e04ad", "license": "Apache-2.0",
        "interface": "pytesseract.image_to_data", "role": "optional OCR baseline",
    },
    "table_transformer_detection": {
        "enabled_env": "CV_ENABLE_TABLE_TRANSFORMER", "model_id": "microsoft/table-transformer-detection",
        "revision": "2357cbe2b5a5d1c03e54f32764f06058933b65ab", "license": "MIT model card",
        "processor": "AutoImageProcessor", "model_class": "AutoModelForObjectDetection", "trust_remote_code": False,
    },
    "table_transformer_structure": {
        "enabled_env": "CV_ENABLE_TABLE_TRANSFORMER", "model_id": "microsoft/table-transformer-structure-recognition-v1.1-all",
        "revision": "7587a7ef111d9dcbf8ac695f1376ab7014340a0c", "license": "MIT model card",
        "processor": "AutoImageProcessor", "model_class": "AutoModelForObjectDetection", "trust_remote_code": False,
    },
    "paddleocr_vl_1_6": {
        "enabled_env": "CV_ENABLE_PADDLEOCR_VL", "model_id": "PaddlePaddle/PaddleOCR-VL-1.6",
        "revision": "c5630abae1d940eafe0697512a0325494b02ab42", "license": "Apache-2.0 model card",
        "sdk": "PaddleOCRVL", "pipeline_version": "v1.6", "maturity": "emerging_2026",
    },
}


def optional_readiness(manifest: dict, artifact_hashes: dict[str, str] | None = None) -> dict:
    hashes = artifact_hashes or {}
    required = ["license"]
    provenance_complete = all(manifest.get(key) for key in required) and bool(hashes)
    return {"enabled": os.getenv(manifest["enabled_env"], "0") == "1", "artifact_hash_status": "recorded" if hashes else "missing",
            "production_provenance_complete": provenance_complete, "comparison_eligible": provenance_complete}


optional_model_observations = {name: {**manifest, **optional_readiness(manifest)} for name, manifest in OPTIONAL_TOOL_MANIFESTS.items()}
assert not any(record["comparison_eligible"] for record in optional_model_observations.values())
pd.DataFrame(optional_model_observations).T[["role", "model_id", "version", "license", "enabled", "artifact_hash_status", "comparison_eligible"]].fillna("—")


In [ ]:
def run_tesseract_adapter(image: Image.Image) -> list[dict]:
    if os.getenv("CV_ENABLE_TESSERACT", "0") != "1":
        return [{"status": "disabled", "reason": "set CV_ENABLE_TESSERACT=1 in an isolated environment"}]
    import pytesseract
    if "5.5.3" not in str(pytesseract.get_tesseract_version()):
        raise RuntimeError("Expected Tesseract 5.5.3; record and review any version change")
    frame = pytesseract.image_to_data(image, output_type=pytesseract.Output.DATAFRAME)
    return frame.dropna(subset=["text"]).to_dict(orient="records")


def run_table_transformer_adapter(image: Image.Image, structure=False):
    if os.getenv("CV_ENABLE_TABLE_TRANSFORMER", "0") != "1":
        return {"status": "disabled", "reason": "set CV_ENABLE_TABLE_TRANSFORMER=1 in an isolated environment"}
    import torch
    from transformers import AutoImageProcessor, AutoModelForObjectDetection
    key = "table_transformer_structure" if structure else "table_transformer_detection"
    manifest = OPTIONAL_TOOL_MANIFESTS[key]
    processor = AutoImageProcessor.from_pretrained(manifest["model_id"], revision=manifest["revision"], trust_remote_code=False)
    model = AutoModelForObjectDetection.from_pretrained(manifest["model_id"], revision=manifest["revision"], trust_remote_code=False).eval()
    inputs = processor(images=image, return_tensors="pt")
    with torch.inference_mode(): outputs = model(**inputs)
    return {"status": "observed_optional", "logits_shape": tuple(outputs.logits.shape), "revision": manifest["revision"]}


def run_paddleocr_vl_adapter(image_path: str):
    if os.getenv("CV_ENABLE_PADDLEOCR_VL", "0") != "1":
        return {"status": "disabled", "reason": "set CV_ENABLE_PADDLEOCR_VL=1 in a governed isolated environment"}
    from paddleocr import PaddleOCRVL
    pipeline = PaddleOCRVL(pipeline_version="v1.6")
    return {"status": "observed_optional", "raw_result": list(pipeline.predict(image_path)),
            "revision": OPTIONAL_TOOL_MANIFESTS["paddleocr_vl_1_6"]["revision"]}


optional_adapter_status = {
    "tesseract": run_tesseract_adapter(render_page(invoice_a, 1)[0]),
    "table_transformer": run_table_transformer_adapter(render_page(invoice_a, 1)[0]),
    "paddleocr_vl": run_paddleocr_vl_adapter("disabled-no-file"),
}
optional_adapter_status


## 19. Evidence artifact

The final record separates locally measured evidence, optional downloaded-model observations, and unresolved production assumptions. It stores no raw enterprise document and grants no action authority.


In [ ]:
def frame_records(frame: pd.DataFrame) -> list[dict]:
    return json.loads(frame.to_json(orient="records"))


failure_taxonomy = {
    "OCR Failure": "recognition/detection mismatch",
    "Layout Failure": "wrong class or region",
    "Reading-Order Failure": "wrong region sequence",
    "Table Detection Failure": "table missing or wrong box",
    "Table Structure Failure": "row/column/span mismatch",
    "Key-Value Binding Failure": "correct strings, wrong relation",
    "Checkbox/Visual-State Failure": "wrong visual state; authenticity not inferred",
    "Normalization Failure": "non-replayable or ambiguous transform",
    "Page Binding Failure": "page reference absent",
    "Provenance Failure": "evidence relation invalid",
    "Unsupported Extraction": "value absent from span/cell evidence",
    "Cross-Page Merge Failure": "continuation relation invalid",
    "Template Shift": "held-out source degradation",
}

evidence_artifact = {
    "course": "Intermediate 03 — Document Intelligence",
    "artifact_type": "document_intelligence_evidence",
    "created_utc": "2026-09-07T00:00:00Z",
    "authorization": "none",
    "document_contract": {"coordinate_format": "xyxy", "origin": "top_left", "unit": "pixel", "document_split_before_pages": True,
                          "source_contract": SOURCE_CONTRACT, "content_hash": "SHA256"},
    "ocr_metrics": frame_records(ocr_clean_results),
    "layout_metrics": frame_records(layout_summary),
    "reading_order_metrics": frame_records(reading_order_metrics),
    "table_metrics": {"clean": table_metrics_clean, "merged_cell_failure": table_metrics_broken_span},
    "field_metrics": frame_records(binding_results),
    "provenance_metrics": frame_records(provenance_failure_attribution),
    "template_shift": frame_records(template_shift),
    "perturbation_results": frame_records(perturbation_summary),
    "review_policy": {"frozen_policy": frozen_evaluation_policy, "review_rate_by_source": review_rate_by_source.to_dict(),
                      "conflicting_value_example": conflicting_value_example},
    "locally_measured_evidence": {
        "ocr_metrics": frame_records(ocr_clean_results),
        "layout_metrics": frame_records(layout_summary),
        "reading_order_metrics": frame_records(reading_order_metrics),
        "table_metrics": {"clean": table_metrics_clean, "merged_cell_failure": table_metrics_broken_span},
        "field_metrics": frame_records(binding_results),
        "provenance_metrics": frame_records(provenance_failure_attribution),
        "template_shift": frame_records(template_shift),
        "perturbation_results": frame_records(perturbation_summary),
        "review_policy": {"frozen_policy": frozen_evaluation_policy, "review_rate_by_source": review_rate_by_source.to_dict()},
    },
    "failure_taxonomy": failure_taxonomy,
    "optional_model_observations": optional_model_observations,
    "unresolved_production_assumptions": [
        "local proxies are not real OCR/layout/model benchmarks",
        "synthetic templates do not represent handwriting, multilingual scripts, or hostile real PDFs",
        "optional artifact hashes and target-hardware measurements are not recorded",
        "malware scanning, tenant isolation, retention, reviewer identity, and downstream authorization are not implemented",
    ],
    "notice": DEMONSTRATION_THRESHOLD_NOTICE,
}
required_artifact_keys = {"course", "artifact_type", "authorization", "document_contract", "ocr_metrics", "layout_metrics",
                          "reading_order_metrics", "table_metrics", "field_metrics", "provenance_metrics", "template_shift",
                          "perturbation_results", "failure_taxonomy", "review_policy", "locally_measured_evidence",
                          "optional_model_observations", "unresolved_production_assumptions"}
assert required_artifact_keys <= evidence_artifact.keys()
assert evidence_artifact["authorization"] == "none"
artifact_dir = Path(".artifacts")
artifact_dir.mkdir(exist_ok=True)
artifact_path = artifact_dir / "intermediate-03-document-intelligence-evidence.json"
artifact_path.write_text(json.dumps(evidence_artifact, indent=2), encoding="utf-8")
{"artifact": str(artifact_path), "bytes": artifact_path.stat().st_size, "top_level_keys": sorted(evidence_artifact)}


## 20. Production upgrade path

![Enterprise document processing separates intake, parsing, verification, review, and authorization.](assets/enterprise-document-architecture.svg)

| Teaching boundary | Production upgrade | Evidence required |
| --- | --- | --- |
| generated images | hardened PDF/image intake and render service | format allow-list, malware results, page/pixel/time limits, sandbox telemetry |
| local OCR/layout proxies | pinned OCR and layout services | model/data/license lineage, source slices, confidence reliability, latency and cost |
| deterministic table schema | detector + structure recognizer + OCR reconciliation | cell/span metrics, continued-table errors, manual failure review |
| fixed key aliases | versioned schema registry and relation model | schema coverage, ambiguity/review rate, rollback tests |
| local JSON artifact | versioned object store and provenance graph | immutable IDs, access logs, encryption, retention/deletion propagation |
| simulated review state | tenant-aware review application | reviewer identity, reason, correction time, disagreement and audit events |
| `authorization = none` | separate policy/action service | least privilege, approvals, idempotency, reconciliation, rollback |

Treat document content as data, never as system instructions. Do not execute embedded scripts, macros, links, or attachments. Minimize retention of raw pages and sensitive crops while keeping enough governed evidence to reproduce an approved decision.


## 21. Exercises and summary

1. Implement a percentage normalizer with a recorded scale assumption and replay test.
2. Add projected row headers without flattening the table schema.
3. Inject a low-CER error in a critical total and explain why corpus CER hides field risk.
4. Design a digital-PDF route that compares embedded text with page rendering before OCR.
5. Define a retention policy for originals, page renders, cropped evidence, structured fields, and reviewer corrections.

You should now be able to explain why OCR text without coordinates is incomplete evidence; why reading order, table structure, field binding, and normalization are distinct predictions; why a correct value can still be unsupported; and why document intelligence is the provenance-preserving input layer for **Intermediate 04 — Multimodal Retrieval & RAG**.
